# Task B judge study: label the Gemini 3.8 Flash segments with candidate LLM judges

Task B annotators label every segment of the **36 Task A windows** (segmented by **Gemini 3.8 Flash**, 947 segments)
with one action: CONSTRUCT, INSPECT, REVISE, REDIRECT or NONACTION. This notebook gives the **same segments** to one
candidate LLM judge at a time, so its labels can be compared with the annotators' and the best judge chosen,
as in Task A.

**Protocol = the package's labeling stage (`05_semantics`)**, applied to fixed segments:
* one request per segment; the system prompt is `LABELING_PROMPT` from **reasoning-anatomy** (read from your package
  zip, so it is exactly the prompt of the main pipeline; an embedded copy of v1.6.1 is the fallback);
* the request holds the problem, the **earlier** segments only (never later text; the grey context before the window
  counts as earlier text) and the segment under review, rendered with the package's `render_user_message`;
* the judge answers the four yes/no tests; the **first yes** decides the action
  (replaces_approach → REDIRECT, repairs_content → REVISE, assesses → INSPECT, adds_content → CONSTRUCT, none → NONACTION).

Every response is saved on Drive (resumable: re-running a cell never pays twice), the **cost** is computed from the
provider's token counts, and `TaskB/judges/<judge>.json` is written. Load those files on the Task B researcher page
(**Agreement → Judges**): it compares every judge with the annotators and names the best validated judge.

**Run order:** sections 1–4 in every session, then one cell of section 5 per judge (any order), then section 6.
Before the first run, check the model IDs and prices in section 3 against your pricing pages.

## 1. Settings

In [ ]:
RB          = "/content/drive/MyDrive/Reasoning Behavior"       # your project folder on Drive
PACKAGE_ZIP = f"{RB}/reasoning-anatomy-v1.8.9.zip"               # read only for its labeling prompt (prompts.py)
WORK        = f"{RB}/TaskB/judge_study"                          # every response, cost and log
OUT_DIR     = f"{RB}/TaskB/judges"                               # <judge>.json for the researcher page
TRACES_URL  = "https://amirhosseinyari2002.github.io/trace-task-b/traces.js"   # the windows and segments annotators see
CONTEXT_CHARS = 1600     # earlier text given with each segment (nearest first; about 400 tokens, the package default)
CONTEXT_SEGMENTS = 24    # and at most this many earlier segments (package default)
CONCURRENCY = 8          # parallel requests
MAX_RETRIES = 4

import os, sys, json, re, time, hashlib, datetime, csv, random, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if not IN_COLAB:                                                   # local test run
    RB = os.environ.get("TASKB_RB", os.path.abspath("taskB_local"))
    WORK, OUT_DIR = f"{RB}/TaskB/judge_study", f"{RB}/TaskB/judges"
    PACKAGE_ZIP = os.environ.get("TASKB_PACKAGE_ZIP", PACKAGE_ZIP)
    TRACES_URL = os.environ.get("TASKB_TRACES", TRACES_URL)

## 2. Setup (once per Colab session)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "google-genai", "anthropic"], check=True)
    from google.colab import userdata
    for k in ("GEMINI_API_KEY", "ANTHROPIC_API_KEY"):                 # Colab: key icon -> add the secret
        try: os.environ[k] = userdata.get(k)
        except Exception: print(f"no {k} secret: judges of that provider are skipped until you add it (key icon)")
os.makedirs(WORK, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
print("work:", WORK, "\nout: ", OUT_DIR)

## 3. Windows, segments and the labeling prompt

The windows are read from the Task B website (`traces.js`), the exact text and segment boundaries annotators label,
and checked against their hashes. A copy is kept in `TaskB/judge_study/traces.snapshot.json`.

In [ ]:
import urllib.request, zipfile, importlib.util

def read_traces(src):
    raw = open(src, encoding="utf-8").read() if os.path.exists(src) else urllib.request.urlopen(src, timeout=60).read().decode("utf-8")
    k = raw.index("window.TRACES = ") + len("window.TRACES = ")
    src_meta = json.loads(re.search(r"window\.TASKB_SOURCE = (\{.*?\});", raw).group(1))
    return json.loads(raw[k: raw.rindex(";")]), src_meta

WINDOWS, SOURCE = read_traces(TRACES_URL)
for t in WINDOWS:
    assert hashlib.sha256(t["text"].encode("utf-8")).hexdigest() == t["sha256"], t["id"] + ": text hash"
    assert hashlib.sha256(json.dumps(t["segs"]).encode()).hexdigest() == t["seg_sha256"], t["id"] + ": segmentation hash"
    assert t["segs"][0][1] == 0 and t["segs"][-1][2] == len(t["text"]), t["id"]
json.dump({"source": SOURCE, "windows": WINDOWS}, open(f"{WORK}/traces.snapshot.json", "w"), ensure_ascii=False)
N_SEG = sum(len(t["segs"]) for t in WINDOWS)
print(f"{len(WINDOWS)} windows, {N_SEG} segments, segmentation: {SOURCE['judge']}")

# ---- the labeling prompt: from the package (exactly the main pipeline's), else the embedded copy ----
EMBEDDED_PROMPT = "<role>\nYou annotate the reasoning of a model that solves math problems, for a research study.\n</role>\n\n<task>\nDetermine the operation the segment under review performs on the reasoning state at that point, by answering the four tests below. Use the earlier segments only to know the state before it; never use later text.\n</task>\n\n<definitions>\nReasoning state: the CONTENT stated so far (claims, values, intermediate results, candidate answers) and the APPROACH being followed (method, plan, search direction).\n\nAnswer four yes/no tests about the segment under review. Answer every test independently.\n1. replaces_approach: does the segment abandon or replace the approach, because the method, plan or reading of the problem is judged wrong, unworkable or unproductive, so that the work continues on a different line?\n   Yes: switching to a different method; abandoning the current line of attack; backtracking to an earlier point to start a different line; restarting after realizing the problem was misread.\n   No: the next step, case or subgoal of the current plan; trying another value, term or variant within the same method; repairing a local error and continuing; rejecting a proposal or doubting an assumption without starting a different line (\"Hmm, maybe not.\", \"Maybe my initial assumption is wrong.\"), which is assessment.\n2. repairs_content: does the segment correct earlier content, i.e. replace, withdraw or restrict it, while the approach continues?\n   Yes: a corrected value or step (\"No, a10 = ...\"); new content that replaces or contradicts earlier content, even without saying the earlier content was wrong (\"Actually, the total number of sites is 400.\"); withdrawing a claim (\"Wait, no, because P can be any polynomial\"); restricting a claim to the cases where it holds; a diagnosis together with its correction. This holds even when the segment starts with \"Wait\" or \"No\".\n   No: simplifying, expanding or rewriting an expression into an equivalent form; continuing a calculation; re-deriving a result and getting the same result; naming an error without correcting it (\"Perhaps I made a mistake.\").\n3. assesses: does the segment evaluate earlier content or the approach?\n   Yes: checking a claim by recomputation, substitution, a test case or a constraint; recognizing an inconsistency; questioning a stated claim; stating a verdict (\"which matches\", \"Not helpful.\"); naming an error; judging progress (\"This is getting complicated.\").\n   No: a plan or announcement of what to do next (\"Let's think in turns.\"); restating values or givens; proposing a new idea, even as a question.\n4. adds_content: does the segment add task-relevant content?\n   Yes: a setup, representation, recalled fact, restated given used in the next step, deduction, calculation, simplification, hypothesis, candidate answer, plan or subgoal; a new idea even when phrased as a question.\n   No: filler, formatting, verbatim repetition, or an announcement the segment does not carry out (\"Let me check.\").\n\nTests describe function, not wording or correctness: a wrong deduction still adds content and a failed check still assesses. Cue words such as \"wait\", \"no\", \"check\" or \"instead\" never decide a test on their own.\n</definitions>\n\n<output>\nsegment_id: the id of the segment under review, unchanged.\nreplaces_approach, repairs_content, assesses, adds_content: true or false, one answer per test.\n</output>"
EMBEDDED_VERSION = "codebook_prompts_v38" + " (embedded copy of reasoning-anatomy 1.6.1)"
EMBEDDED_TAGS = {"question": "problem", "previous_segments": "earlier_segments", "local_context": "referenced_segment", "target": "segment_under_review"}
EMBEDDED_FINAL = "Answer the four tests for the segment under review."
def _embedded_render(task, data_json):
    data, parts = json.loads(data_json), []
    for key, value in data.items():
        if key == "local_context" and not value: continue
        tag = EMBEDDED_TAGS.get(key, key)
        if key == "previous_segments":
            body = "\n".join("- " + " ".join(str(v.get("text", "")).split()) for v in value)
        elif isinstance(value, dict) and "text" in value:
            body = f"[{value['segment_id']}] " + " ".join(str(value["text"]).split())
        else:
            body = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False, separators=(",", ":"))
        parts.append(f"<{tag}>\n{body}\n</{tag}>")
    parts.append(f"<instruction>{EMBEDDED_FINAL}</instruction>")
    return "\n".join(parts)

PROMPT_SOURCE = "embedded"
SYSTEM, PROMPT_VERSION, render_user_message = EMBEDDED_PROMPT, EMBEDDED_VERSION, _embedded_render
if os.path.exists(PACKAGE_ZIP):
    with zipfile.ZipFile(PACKAGE_ZIP) as z:
        names = [n for n in z.namelist() if n.endswith("src/reasoning_anatomy/prompts.py")]
        init = [n for n in z.namelist() if n.endswith("src/reasoning_anatomy/__init__.py")]
        if names:
            os.makedirs("/tmp/ra_prompts", exist_ok=True)
            p = z.extract(names[0], "/tmp/ra_prompts")
            spec = importlib.util.spec_from_file_location("ra_prompts", p); mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
            ver = re.search(r'__version__ = "([^"]+)"', z.read(init[0]).decode()).group(1) if init else "?"
            SYSTEM, render_user_message = mod.LABELING_PROMPT, mod.render_user_message
            PROMPT_VERSION, PROMPT_SOURCE = f"{mod.PROMPT_VERSION} (reasoning-anatomy {ver})", f"package {ver}"
PROMPT_SHA = hashlib.sha256(SYSTEM.encode()).hexdigest()
print("labeling prompt:", PROMPT_VERSION, "| source:", PROMPT_SOURCE, "| sha256", PROMPT_SHA[:12])
if PROMPT_SOURCE == "embedded": print("  (package zip not found: using the embedded copy; put the zip at PACKAGE_ZIP to use your version)")

TESTS = ["replaces_approach", "repairs_content", "assesses", "adds_content"]
ORDER = [("replaces_approach", "REDIRECT"), ("repairs_content", "REVISE"), ("assesses", "INSPECT"), ("adds_content", "CONSTRUCT")]
def action(t):                                    # the package's rule: the first test answered yes decides
    for k, a in ORDER:
        if t[k]: return a
    return "NONACTION"
SCHEMA = {"type": "object", "properties": {"segment_id": {"type": "integer"}, **{k: {"type": "boolean"} for k in TESTS}},
          "required": ["segment_id", *TESTS], "additionalProperties": False}

def earlier_units(t):                             # grey context paragraphs, then the window's segments
    ctx = [p.strip() for p in re.split(r"\n\s*\n", t.get("context", "").lstrip("… ").strip()) if p.strip()]
    return ctx
def build_requests():
    reqs = []
    for t in WINDOWS:
        ctx, segs = earlier_units(t), [t["text"][a:b] for _, a, b in t["segs"]]
        for k, seg in enumerate(segs):
            earlier = [(-(len(ctx) - j), c) for j, c in enumerate(ctx)] + [(i, segs[i]) for i in range(k)]
            chosen, used = [], 0
            for sid, txt in reversed(earlier):            # nearest first, within the character and segment budget
                txt = " ".join(txt.split())
                if not txt: continue
                if chosen and (used + len(txt) > CONTEXT_CHARS or len(chosen) >= CONTEXT_SEGMENTS): break
                chosen.append({"segment_id": max(sid, 0), "text": txt}); used += len(txt)
            payload = {"question": t["question"], "previous_segments": list(reversed(chosen)),
                       "target": {"segment_id": k, "text": " ".join(seg.split()) or seg}}
            user = render_user_message("annotation", json.dumps(payload, ensure_ascii=False, separators=(",", ":")))
            reqs.append({"item_id": t["id"], "seg": k, "user": user})
    return reqs
REQUESTS = build_requests()
print(len(REQUESTS), "requests per judge; example:\n" + "-" * 60 + "\n" + REQUESTS[1]["user"][:1200])

## 4. Candidate judges and the callers (edit the judges here)

Prices are USD per 1M tokens (standard paid tier; output includes thinking tokens), as in the Task A judge study.
Gemini: `thinking_level` (3.x / 4: lowest is *minimal* for Flash-Lite, *low* for 3.8 and 4) or `thinking_budget`
(2.5: 0 = off). Claude: `thinking` *disabled* (allowed at effort low–high) or *adaptive*, `effort` low/medium/high;
the fixed system prompt is cached (`cache_control`), and cached input is billed at Anthropic's cache prices.

In [ ]:
JUDGES = {
  "g3.8-flash":             {"provider": "gemini", "model": "gemini-3.8-flash", "thinking_level": "low", "max_output_tokens": 4096, "price": [0.75, 3.75],
                             "note": "the Task A segmentation judge; introductory price to 2026-12-31, then 1.50 / 7.50"},
  "g3.5-flash-lite":        {"provider": "gemini", "model": "gemini-3.5-flash-lite", "thinking_level": "minimal", "max_output_tokens": 1024, "price": [0.30, 2.50]},
  "g3.5-flash-lite-think":  {"provider": "gemini", "model": "gemini-3.5-flash-lite", "thinking_level": "low", "max_output_tokens": 4096, "price": [0.30, 2.50]},
  "g3.1-flash-lite":        {"provider": "gemini", "model": "gemini-3.1-flash-lite", "thinking_level": "minimal", "max_output_tokens": 1024, "price": [0.25, 1.50]},
  "g2.5-flash-lite":        {"provider": "gemini", "model": "gemini-2.5-flash-lite", "thinking_budget": 0, "max_output_tokens": 1024, "price": [0.10, 0.40]},
  "g4-hydrogen":            {"provider": "gemini", "model": "gemini-4-hydrogen", "thinking_level": "low", "max_output_tokens": 4096, "price": [1.50, 7.50]},
  "claude-haiku-5.5":       {"provider": "claude", "model": "claude-haiku-5-5", "thinking": "disabled", "effort": "low", "max_output_tokens": 1024, "price": [0.10, 0.50]},
  "claude-haiku-5.5-think": {"provider": "claude", "model": "claude-haiku-5-5", "thinking": "adaptive", "effort": "low", "max_output_tokens": 8192, "price": [0.10, 0.50]},
  "mock":                   {"provider": "mock", "model": "mock", "price": [0, 0], "note": "no API call: tests the pipeline"},
}

_clients = {}
def gemini():
    if "g" not in _clients:
        from google import genai
        _clients["g"] = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
    return _clients["g"]
def claude():
    if "c" not in _clients:
        import anthropic
        _clients["c"] = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
    return _clients["c"]

def call_gemini(cfg, user):
    from google.genai import types
    th = None
    if cfg.get("thinking_level"): th = types.ThinkingConfig(thinking_level=cfg["thinking_level"])
    elif cfg.get("thinking_budget") is not None: th = types.ThinkingConfig(thinking_budget=cfg["thinking_budget"])
    kw = dict(system_instruction=SYSTEM, response_mime_type="application/json", max_output_tokens=cfg.get("max_output_tokens", 1024))
    if th is not None: kw["thinking_config"] = th
    try: conf = types.GenerateContentConfig(response_json_schema=SCHEMA, **kw)
    except Exception:                                   # older google-genai: OpenAPI-style schema
        conf = types.GenerateContentConfig(response_schema={k: v for k, v in SCHEMA.items() if k != "additionalProperties"}, **kw)
    r = gemini().models.generate_content(model=cfg["model"], contents=user, config=conf)
    u = r.usage_metadata
    usage = {"input": u.prompt_token_count or 0, "cached": getattr(u, "cached_content_token_count", 0) or 0,
             "output": u.candidates_token_count or 0, "thinking": getattr(u, "thoughts_token_count", 0) or 0}
    pin, pout = cfg["price"]
    usage["cost"] = (usage["input"] * pin + (usage["output"] + usage["thinking"]) * pout) / 1e6
    return r.text, usage

def call_claude(cfg, user):
    r = claude().messages.create(
        model=cfg["model"], max_tokens=cfg.get("max_output_tokens", 1024),
        system=[{"type": "text", "text": SYSTEM, "cache_control": {"type": "ephemeral"}}],
        messages=[{"role": "user", "content": user}],
        thinking={"type": cfg.get("thinking", "disabled")},
        output_config={"effort": cfg.get("effort", "low"), "format": {"type": "json_schema", "schema": SCHEMA}})
    if r.stop_reason == "refusal": raise RuntimeError("refusal")
    if r.stop_reason == "max_tokens": raise RuntimeError("max_tokens reached")
    text = next(b.text for b in r.content if b.type == "text")
    u = r.usage
    usage = {"input": u.input_tokens, "cache_read": u.cache_read_input_tokens or 0, "cache_write": u.cache_creation_input_tokens or 0,
             "output": u.output_tokens}                 # output_tokens include thinking
    pin, pout = cfg["price"]
    usage["cost"] = (usage["input"] * pin + usage["cache_write"] * pin * 1.25 + usage["cache_read"] * pin * 0.1 + usage["output"] * pout) / 1e6
    return text, usage

def call_mock(cfg, user):
    k = int(re.search(r"<segment_under_review>\n\[(\d+)\]", user).group(1))
    h = int(hashlib.sha1(user.encode()).hexdigest(), 16)
    t = {"replaces_approach": h % 23 == 0, "repairs_content": h % 19 == 0, "assesses": h % 3 == 0, "adds_content": h % 7 != 0}
    return json.dumps({"segment_id": k, **t}), {"input": len(user) // 4, "output": 20, "cost": 0.0}

CALLERS = {"gemini": call_gemini, "claude": call_claude, "mock": call_mock}
print("judges:", [k for k in JUDGES])

## 5. Run the judges: one cell per judge

Each cell first sends **one** request (a few cents; a wrong model ID or key stops here), then labels all segments.
Responses are appended to `TaskB/judge_study/<judge>/responses.jsonl` as they arrive; re-running the cell resumes and
only retries what is missing or failed. At the end it writes `TaskB/judges/<judge>.json` and adds a row to
`TaskB/judge_study/costs.csv`.

In [ ]:
_lock = threading.Lock()
def req_key(name, cfg, r):
    spec = json.dumps({k: v for k, v in cfg.items() if k not in ("note", "price")}, sort_keys=True)
    return hashlib.sha256((spec + PROMPT_SHA + r["user"]).encode()).hexdigest()

def parse(text, seg):
    d = json.loads(text)
    if any(not isinstance(d.get(k), bool) for k in TESTS): raise ValueError("a test is not true/false")
    if int(d.get("segment_id", -1)) != seg: raise ValueError(f"segment_id {d.get('segment_id')} != {seg}")
    return {k: d[k] for k in TESTS}

def one(name, cfg, r, path):
    err = None
    for attempt in range(MAX_RETRIES):
        try:
            text, usage = CALLERS[cfg["provider"]](cfg, r["user"])
            tests = parse(text, r["seg"])
            rec = {"key": r["key"], "item_id": r["item_id"], "seg": r["seg"], "ok": True, "tests": tests, "action": action(tests),
                   "usage": usage, "model": cfg["model"], "ts": datetime.datetime.utcnow().isoformat() + "Z"}
            break
        except Exception as e:                         # rate limits, timeouts, invalid output: back off and retry
            err = f"{type(e).__name__}: {e}"[:300]; time.sleep(min(60, 2 ** attempt + random.random()))
    else:
        rec = {"key": r["key"], "item_id": r["item_id"], "seg": r["seg"], "ok": False, "error": err, "model": cfg["model"],
               "ts": datetime.datetime.utcnow().isoformat() + "Z"}
    with _lock, open(path, "a", encoding="utf-8") as f: f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    return rec

def run_judge(name):
    cfg = JUDGES[name]
    if cfg["provider"] == "gemini" and not os.environ.get("GEMINI_API_KEY"): raise SystemExit("GEMINI_API_KEY is not set (key icon, then section 2)")
    if cfg["provider"] == "claude" and not os.environ.get("ANTHROPIC_API_KEY"): raise SystemExit("ANTHROPIC_API_KEY is not set (key icon, then section 2)")
    d = f"{WORK}/{name}"; os.makedirs(d, exist_ok=True); path = f"{d}/responses.jsonl"
    reqs = [dict(r, key=req_key(name, cfg, r)) for r in REQUESTS]
    done = {}
    if os.path.exists(path):
        for line in open(path, encoding="utf-8"):
            try: rec = json.loads(line)
            except ValueError: continue
            if rec.get("ok"): done[rec["key"]] = rec
    todo = [r for r in reqs if r["key"] not in done]
    print(f"{name}: {len(reqs) - len(todo)} of {len(reqs)} segments already labeled, {len(todo)} to do")
    if todo and not done:                                   # check the judge with one request first
        rec = one(name, cfg, todo[0], path)
        if not rec["ok"]: raise SystemExit(f"the first request failed, nothing else was sent: {rec['error']}")
        done[rec["key"]] = rec; todo = todo[1:]
        print(f"check ok: {rec['action']}  usage {rec['usage']}")
    t0, n_fail = time.time(), 0
    with ThreadPoolExecutor(CONCURRENCY) as ex:
        futs = [ex.submit(one, name, cfg, r, path) for r in todo]
        for k, fu in enumerate(as_completed(futs), 1):
            rec = fu.result()
            if rec["ok"]: done[rec["key"]] = rec
            else: n_fail += 1
            if k % 100 == 0 or k == len(futs): print(f"  {k}/{len(futs)}  failed {n_fail}  {time.time() - t0:.0f}s")
    return write_judge(name, cfg, reqs, done)

def write_judge(name, cfg, reqs, done):
    labels, tests, missing = {}, {}, 0
    for t in WINDOWS:
        labels[t["id"]], tests[t["id"]] = [], []
        for k in range(len(t["segs"])):
            rec = done.get(next(r["key"] for r in reqs if r["item_id"] == t["id"] and r["seg"] == k))
            labels[t["id"]].append(rec["action"] if rec else None)
            tests[t["id"]].append([rec["tests"][x] for x in TESTS] if rec else None)
            missing += rec is None
    recs = list(done.values())
    tok = {k: sum(r["usage"].get(k, 0) for r in recs) for k in ("input", "cached", "cache_read", "cache_write", "output", "thinking")}
    cost = sum(r["usage"].get("cost", 0) for r in recs)
    out = {"format": "trace-task-b-judge/v1", "judge": name, "provider": cfg["provider"], "model": cfg["model"],
           "settings": {k: v for k, v in cfg.items() if k not in ("note",)},
           "protocol": "reasoning-anatomy labeling prompt, one request per fixed segment, earlier context only; first yes of "
                       "replaces_approach/repairs_content/assesses/adds_content decides REDIRECT/REVISE/INSPECT/CONSTRUCT, none = NONACTION",
           "prompt_version": PROMPT_VERSION, "prompt_sha256": PROMPT_SHA, "context_chars": CONTEXT_CHARS, "context_segments": CONTEXT_SEGMENTS,
           "segmentation": SOURCE, "seg_sha256": {t["id"]: t["seg_sha256"] for t in WINDOWS},
           "labels": labels, "tests_order": TESTS, "tests": tests, "missing": missing,
           "requests": len(recs), "tokens": tok, "cost_usd": round(cost, 4), "created": datetime.datetime.utcnow().isoformat() + "Z"}
    json.dump(out, open(f"{OUT_DIR}/{name}.json", "w"), ensure_ascii=False)
    new = not os.path.exists(f"{WORK}/costs.csv")
    with open(f"{WORK}/costs.csv", "a", newline="") as f:
        w = csv.writer(f)
        if new: w.writerow(["time", "judge", "model", "segments_labeled", "missing", "input_tokens", "output_tokens", "thinking_tokens", "cost_usd"])
        w.writerow([out["created"], name, cfg["model"], len(recs), missing, tok["input"], tok["output"], tok["thinking"], out["cost_usd"]])
    dist = {}
    for v in labels.values():
        for x in v: dist[x] = dist.get(x, 0) + 1
    print(f"wrote {OUT_DIR}/{name}.json: {len(recs)} labeled, {missing} missing (re-run the cell to retry), ${cost:.3f}; labels {dist}")
    return out

In [ ]:
run_judge("g3.8-flash")

In [ ]:
run_judge("g3.5-flash-lite")

In [ ]:
run_judge("claude-haiku-5.5")

In [ ]:
run_judge("claude-haiku-5.5-think")

Optional judges (cheaper, older, thinking variants, frontier):

In [ ]:
# run_judge("g2.5-flash-lite")
# run_judge("g3.5-flash-lite-think")
# run_judge("g3.1-flash-lite")
# run_judge("g4-hydrogen")

Pipeline test without any API call (writes `TaskB/judges/mock.json`; do not load it on the researcher page):

In [ ]:
# run_judge("mock")

## 6. Costs, label distributions, and judge–judge agreement

The decision (which judge agrees best with the annotators) is made on the **researcher page**: load the files in
`TaskB/judges/` under **Agreement → Judges** (with `TaskA/private/key.csv` for the per-model claim J2).
This section only summarizes the runs.

In [ ]:
import pandas as pd
LAB = ["CONSTRUCT", "INSPECT", "REVISE", "REDIRECT", "NONACTION"]
files = sorted(p for p in Path(OUT_DIR).glob("*.json") if p.stem != "mock")
J = {p.stem: json.load(open(p)) for p in files}
if not J: raise SystemExit("no judge file yet: run a judge in section 5")
rows = []
for n, d in J.items():
    flat = [x for v in d["labels"].values() for x in v]
    rows.append({"judge": n, "model": d["model"], "labeled": sum(x is not None for x in flat), "missing": d["missing"], "cost $": d["cost_usd"],
                 **{l: round(flat.count(l) / max(1, len(flat)), 3) for l in LAB}})
display(pd.DataFrame(rows).set_index("judge"))

def kappa(a, b):                                   # Cohen's kappa over segments both labeled
    pairs = [(x, y) for x, y in zip(a, b) if x and y]
    if not pairs: return float("nan")
    po = sum(x == y for x, y in pairs) / len(pairs)
    pe = sum((sum(x == l for x, _ in pairs) / len(pairs)) * (sum(y == l for _, y in pairs) / len(pairs)) for l in LAB)
    return (po - pe) / (1 - pe) if pe < 1 else float("nan")
names = list(J)
flat = {n: [x for t in WINDOWS for x in J[n]["labels"][t["id"]]] for n in names}
display(pd.DataFrame([[round(kappa(flat[a], flat[b]), 3) for b in names] for a in names], index=names, columns=names))
print("Next: Task B researcher page -> Agreement -> Judges: load", ", ".join(p.name for p in files))